> ⚠️ **このノートはColabでは動きません。**
> FreeCAD / Blender 本体（デスクトップアプリ）が必要です。GitHub上では閲覧のみ。コードは各アプリのPythonコンソール／Scriptingに貼り付けて使います。
>
> （Colabで動くのは統計・マーケ系と `SolidPython` のノートです）

# ホビー-02. Blender で陶芸の器をつくる

**Blender** は無料の3DCGソフト。FreeCADより**曲面が自由**で、
ぷっくりした花瓶や有機的な形が得意です。Pythonで自動生成できます。

このノートのコードは Blender の **Scripting** タブに貼り付けて実行します。

> 🏺 **このツールはこんな時に**：**有機的な形や模様**。一点物のアート器、**スタンプ/sprig（型押し飾り）**、クレイプリンタ用の造形。

## 0. 準備

1. [blender.org](https://www.blender.org/) から Blender をインストール
2. 上のタブの **Scripting** をクリック
3. 中央の **New** で新しいテキストを作り、コードを貼り付け
4. ▶ (Run Script) または `Alt+P` で実行

## 1. 考え方：プロファイルを spin（回転）させる

FreeCADと同じ「回転体」ですが、Blenderでは輪郭を**点の鎖**で作り、
`bmesh.ops.spin` でぐるっと回してメッシュ（面の集まり）にします。
あとから厚み・なめらかさを足せるのが強みです。

### 🏺 陶芸ならではの注意：収縮率

粘土は乾燥・焼成で **10〜15%ちぢみます**。焼き上がりで口径120mmにしたいなら、
モデルは `120 ÷ (1 - 0.13) ≈ 138mm` で作ります。これは比の良い練習にもなります。

```python
finish = 120          # 焼き上がりで欲しい口径(mm)
shrink = 0.13         # 収縮率13%
model = finish / (1 - shrink)
print(round(model, 1), 'mm で設計する')
```

## 2. 花瓶をつくるスクリプト

`profile` は `(半径, 高さ)` の点。くびれや膨らみを自由に作れます。

In [ ]:
# === BlenderのScriptingタブに貼り付け ===
import bpy, bmesh, math      # bpy=Blender操作, bmesh=メッシュ編集, math=数学

# 既存オブジェクトを消す
bpy.ops.object.select_all(action='SELECT')   # 全オブジェクトを選択
bpy.ops.object.delete()                       # 削除

# 花瓶の輪郭 (半径, 高さ) ※下から上へ。膨らみとくびれ
profile = [
    (20, 0), (40, 4), (55, 30), (50, 60),   # 下半分がふくらむ
    (30, 90), (28, 110), (40, 130), (38, 140) # 上でくびれて口が開く
]

mesh = bpy.data.meshes.new('Vase')            # 空のメッシュを作る
obj = bpy.data.objects.new('Vase', mesh)      # メッシュを持つオブジェクト
bpy.context.collection.objects.link(obj)      # シーンに追加

bm = bmesh.new()                              # 編集用のメッシュ
verts = [bm.verts.new((r, 0, h)) for r, h in profile]   # 輪郭の各点を頂点にする
edges = [bm.edges.new((verts[i], verts[i+1])) for i in range(len(verts)-1)]  # 隣どうしを線で結ぶ

# Z軸まわりに360°回す（64分割でなめらかに）
geom = verts + edges                          # 回す対象（頂点と辺）
bmesh.ops.spin(bm, geom=geom, cent=(0,0,0), axis=(0,0,1),
               angle=math.radians(360), steps=64)        # 回転体にする
bmesh.ops.remove_doubles(bm, verts=bm.verts, dist=0.001) # 重なった頂点を結合
bm.to_mesh(mesh)                              # 編集結果をメッシュに反映
bm.free()                                     # 後片付け

ここまでで「紙のように薄い」花瓶ができます。次に厚みとなめらかさを足します。

In [ ]:
# 厚み(肉厚)をつける Solidify モディファイア
solid = obj.modifiers.new('thick', 'SOLIDIFY')   # 厚みを足す機能を追加
solid.thickness = 5   # 肉厚5mm

# 表面をなめらかにする
for p in mesh.polygons:           # すべての面を
    p.use_smooth = True           # なめらか表示にする
subsurf = obj.modifiers.new('smooth', 'SUBSURF')   # 細分化でさらになめらかに
subsurf.levels = 2                # 細かさのレベル

print('花瓶ができました！ テンキー1で正面、マウス中ボタンで回転')

## 3. 形をいじって遊ぶ

`profile` の数字を変えるだけで、いろんな器になります。

| 作りたいもの | ヒント |
|---|---|
| 茶碗 | 点を少なく、低く広く。最後の高さを60くらいに |
| 徳利（とっくり） | 真ん中を太く、首を細く（半径を急に小さく） |
| 一輪挿し | 全体を細長く、口をすぼめる |

> 💡 数字を少し変えて ▶ を押す、をくり返すと感覚がつかめます。

## 4. 書き出し

メニュー **File → Export → STL (.stl)** で保存。3Dプリンタや、
石こう型づくりの参考形状として使えます。コードでも出せます。

In [ ]:
# STLで書き出し（パスは自分用に変更）
bpy.ops.object.select_all(action='DESELECT')   # いったん選択を解除
obj.select_set(True)                            # 花瓶だけ選択
bpy.ops.wm.stl_export(filepath='/Users/あなた/Desktop/vase.stl',   # STLで保存
                      export_selected_objects=True)

---
## 🏆 チャレンジ課題

**課題1.** `profile` を増やして、上にいくほど口が開く「ラッパ型の花器」を作ろう。

**課題2.** `solid.thickness` を 2 と 10 で比べ、見た目と「重そうさ」の違いを確かめよう。

**課題3.**（発展）`profile` の半径を `math.sin` で波打たせ、ねじれ・凹凸のある現代的な器に挑戦。

> 次は **SolidPython** で、コードそのもので形を定義する作り方を学びます（03のノートへ）。

## 📦 出力ファイルの見方・操作

**可視化は Blender のビューポート内**で完結します。
- 回転：マウス中ボタンドラッグ／テンキー **1**=正面・**7**=真上
- ズーム：マウスホイール
- 書き出し：メニュー **File → Export → STL**（コードの `wm.stl_export` でも可）
- 💡 Blenderの強み：書き出す前に **スカルプトモード**で手彫りのように凹凸を足せます。

### `.stl` ファイルの見方・使い道（共通）

`.stl` は3D形状の標準フォーマット。次のように扱えます。
- **見るだけ**：ダブルクリックでOSの3Dビューア（Mac=プレビュー / Win=3Dビューア）。または無料オンラインビューア（`viewstl.com` などにファイルをドラッグ）。
- **編集**：Blender や FreeCAD に読み込んで形を調整。
- **3Dプリント**：スライサー（**Cura** / **PrusaSlicer**、無料）で `.stl` を開く → 印刷設定 → プリンタへ。

> 🏺 **陶芸での使い道**：プリントした器そのものは焼けませんが、**石こう型の原型**や、ろくろ・たたら作りの**テンプレート/ゲージ**として活用できます。